# KI-3: Stage-Wise Switching Experiment

## 실험 개요

**연구 질문:** Transmitter의 CoT(Chain-of-Thought) 내부에서 audience-aware 전환 시점이 최종 메시지 품질과 Receiver 정확도에 미치는 영향은?

### 에이전트 구성
- **Tx (Transmitter):** GPT-4o-mini - 3단계 CoT 체이닝으로 문제를 풀고, Stage 3 출력만 전송
- **Rx (Receiver):** GPT-4o-mini - 전송된 메시지에서 최종 답을 추출

### 실험 조건 (5가지)
| 조건 | Tx Stage 1-2 | Tx Stage 3 | Rx |
|------|-------------|------------|----|
| **All General** | General solver | General final | Simple extractor |
| **All Audience-Aware** | Concise expert | Compact final | Expert interpreter |
| **Tx-Only Switch** | Detailed solver | Compress for expert | Simple extractor |
| **Both Switch** | Detailed solver | Compress for expert | Decompress + verify |
| **Reverse** | Concise outline | Detailed solve | Free reader |

### 문제 유형 (15문제)
- **확률 (1-5):** Bayes' theorem, conditional probability, Monty Hall
- **최적화 (6-10):** Fencing, box optimization, linear programming
- **다단계 추론 (11-15):** Clock drift, fly puzzle, locker problem

### 핵심 설계
- Tx는 3-stage CoT 체이닝 수행 (Stage1 -> Stage2 -> Stage3)
- **Stage 3 출력만** Rx에게 전송됨 (intermediate stages는 전송 안 됨)
- 측정: accuracy + message tokens (Stage 3 길이)

### 기대 결과
- Both Switch가 최고 성능 (Tx 압축 + Rx 복원)
- All Audience-Aware는 메시지 토큰 최소이나 정확도 하락 가능
- Reverse는 비효율적 (압축 -> 확장 순서)

In [ ]:
# ============================================================
# Setup: API Key and Dependencies
# ============================================================
import os
import json
import re
import time
import math
from datetime import datetime

import requests
import pandas as pd

# Load API key from environment variable or prompt for input
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY:
    OPENAI_API_KEY = input("OpenAI API Key를 입력하세요: ").strip()

assert OPENAI_API_KEY, "API Key가 설정되지 않았습니다."

MODEL = "gpt-4o-mini"
TEMPERATURE = 0
API_URL = "https://api.openai.com/v1/chat/completions"

print(f"Model: {MODEL}")
print(f"Temperature: {TEMPERATURE}")
print("API Key: OK (loaded)")

In [ ]:
# ============================================================
# Common Functions: API Call, Number Extraction, Token Count
# ============================================================

total_api_tokens = {"prompt": 0, "completion": 0}

def call_llm(system: str, user_msg: str, retries: int = 3) -> dict:
    """Call OpenAI API with retry logic."""
    global total_api_tokens
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {OPENAI_API_KEY}",
    }
    payload = {
        "model": MODEL,
        "temperature": TEMPERATURE,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": user_msg},
        ],
    }
    for attempt in range(retries):
        try:
            resp = requests.post(API_URL, headers=headers, json=payload, timeout=120)
            if resp.status_code == 429:
                wait = (attempt + 1) * 5
                print(f"  Rate limited, waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            data = resp.json()
            usage = data.get("usage", {})
            total_api_tokens["prompt"] += usage.get("prompt_tokens", 0)
            total_api_tokens["completion"] += usage.get("completion_tokens", 0)
            return {"content": data["choices"][0]["message"]["content"], "usage": usage}
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(2)


def extract_number(text: str):
    """Extract the last numeric value from text."""
    cleaned = text.replace(",", "").replace("%", "")
    matches = re.findall(r"-?\d+\.?\d*", cleaned)
    if not matches:
        return None
    return float(matches[-1])


def count_approx_tokens(text: str) -> int:
    """Approximate token count (~4 chars per token)."""
    return math.ceil(len(text) / 4)


def parse_time_to_minutes(text: str):
    """Parse time strings like '5:00 PM' or '17:00' to minutes from midnight."""
    pm_match = re.search(r"(\d{1,2}):(\d{2})\s*(PM|AM)", text, re.IGNORECASE)
    if pm_match:
        h = int(pm_match.group(1))
        m = int(pm_match.group(2))
        if pm_match.group(3).upper() == "PM" and h < 12:
            h += 12
        if pm_match.group(3).upper() == "AM" and h == 12:
            h = 0
        return h * 60 + m
    mil_match = re.search(r"(\d{1,2}):(\d{2})", text)
    if mil_match:
        return int(mil_match.group(1)) * 60 + int(mil_match.group(2))
    return None


print("Common functions loaded.")

## 문제 목록

15개 문제는 3개 카테고리로 나뉩니다:

| ID | Category | Problem Description |
|----|----------|--------------------|
| 1-5 | Probability | Bayes' theorem, conditional probability, Monty Hall |
| 6-10 | Optimization | Fencing problems, box optimization, LP |
| 11-15 | Multi-step | Clock drift, fly puzzle, rope around Earth, lockers, tournament |

In [ ]:
# ============================================================
# Problem Definitions (15 problems: probability, optimization, multi-step)
# ============================================================

PROBLEMS = [
    # Probability (1-5)
    {"id": 1, "category": "probability",
     "text": "A disease has 1% prevalence. A test has 95% sensitivity and 95% specificity. What is P(disease | positive test result)? Give the answer as a percentage.",
     "answer": 16.1, "tolerance_fn": None},
    {"id": 2, "category": "probability",
     "text": "3 urns: A has 5 red & 3 blue balls, B has 2 red & 6 blue, C has 4 red & 4 blue. Pick a random urn, draw one ball, it is red. What is P(urn A was chosen)? Give as percentage.",
     "answer": 45.5, "tolerance_fn": None},
    {"id": 3, "category": "probability",
     "text": 'P(spam)=0.3, P("free"|spam)=0.8, P("free"|not spam)=0.1. What is P(spam|"free")? Give as percentage.',
     "answer": 77.4, "tolerance_fn": None},
    {"id": 4, "category": "probability",
     "text": "Machine1 produces 60% of items with 2% defect rate. Machine2 produces 40% with 5% defect rate. A defective item is found. P(Machine1 | defective)? Give as percentage.",
     "answer": 37.5, "tolerance_fn": None},
    {"id": 5, "category": "probability",
     "text": "Monty Hall problem: you picked door 1, host opens door 3 (goat). What is P(car behind door 2)? Give as percentage.",
     "answer": 66.7, "tolerance_fn": None},
    # Optimization (6-10)
    {"id": 6, "category": "optimization",
     "text": "You have 100m of fence to enclose a rectangular area along a river (river is one side, no fence needed). What is the maximum area in square meters?",
     "answer": 1250, "tolerance_fn": None},
    {"id": 7, "category": "optimization",
     "text": "An open-top box with a square base must have volume 1000 cm^3. What side length of the base minimizes the total surface area? Give in cm.",
     "answer": 10, "tolerance_fn": None},
    {"id": 8, "category": "optimization",
     "text": "Find the global maximum value of f(x) = x^3 - 6x^2 + 9x + 1 on the interval [0, 5].",
     "answer": 21, "tolerance_fn": None},
    {"id": 9, "category": "optimization",
     "text": "A farmer has 200m of fencing to enclose a rectangular field. What is the maximum area in square meters?",
     "answer": 2500, "tolerance_fn": None},
    {"id": 10, "category": "optimization",
     "text": "Product A gives $5 profit and uses 2 hours of labor. Product B gives $8 profit and uses 3 hours. You have 120 hours available. Maximize total profit. What is the max profit in dollars?",
     "answer": 320, "tolerance_fn": None},
    # Multi-step (11-15)
    {"id": 11, "category": "multistep",
     "text": "A clock gains 5 minutes every hour. It is set correctly at noon. What is the real time when the clock shows 6:00 PM? Give answer as HH:MM (in real time).",
     "answer": 300,  # 5*60 = 300 minutes = 5:00 PM
     "tolerance_fn": "time"},  # special handling
    {"id": 12, "category": "multistep",
     "text": "Two trains are 300km apart heading toward each other at 60 km/h and 90 km/h. A fly starts at one train and bounces between them at 120 km/h until they meet. Total distance flown by the fly in km?",
     "answer": 240, "tolerance_fn": None},
    {"id": 13, "category": "multistep",
     "text": "A rope is wrapped tightly around Earth's equator (circumference 40000 km). You add 1 meter of rope. If the rope is lifted uniformly, how high above the surface (in meters, to 3 decimal places)?",
     "answer": 0.159, "tolerance_fn": None},
    {"id": 14, "category": "multistep",
     "text": "100 lockers in a row, all closed. Student 1 opens all. Student 2 toggles every 2nd. Student k toggles every k-th locker, for k=1..100. How many lockers are open at the end?",
     "answer": 10, "tolerance_fn": None},
    {"id": 15, "category": "multistep",
     "text": "A 16-player single-elimination tournament. How many total games are played?",
     "answer": 15, "tolerance_fn": None},
]

print(f"Loaded {len(PROBLEMS)} problems:")
for p in PROBLEMS:
    print(f"  P{p['id']:2d} [{p['category']:12s}] answer={p['answer']}")

In [ ]:
# ============================================================
# Condition Definitions: 5 conditions with different Tx/Rx stage prompts
# ============================================================

CONDITIONS = {
    "all-general": {
        "label": "All General",
        "tx_stages": [
            {"system": "You are a math problem solver. Solve the given problem step by step with full reasoning."},
            {"system": "You are a math problem solver. Continue solving step by step. Build on the previous analysis."},
            {"system": "You are a math problem solver. Provide the final clean solution with the numeric answer clearly stated."},
        ],
        "rx_stages": [
            {"system": "Extract the final numeric answer from the given solution. Output ONLY the number."},
        ],
    },
    "all-audience": {
        "label": "All Audience-Aware",
        "tx_stages": [
            {"system": "You are solving a math problem. The recipient is a math expert. Be extremely concise. Use mathematical notation. No verbose explanations."},
            {"system": "Continue. Stay concise. Math notation only. Expert audience."},
            {"system": "Final answer for math expert. Minimal text. Key steps + answer only."},
        ],
        "rx_stages": [
            {"system": "You are a math expert. Interpret the compact mathematical message. Extract the final numeric answer. Output ONLY the number."},
        ],
    },
    "tx-switch": {
        "label": "Tx-Only Switch",
        "tx_stages": [
            {"system": "You are a math problem solver. Solve step by step with full detailed reasoning."},
            {"system": "Continue with full detailed step-by-step reasoning. Show all work."},
            {"system": "Now compress your solution for a math expert recipient. Include only essential steps and the final answer. Use compact notation."},
        ],
        "rx_stages": [
            {"system": "Extract the final numeric answer from the given message. Output ONLY the number."},
        ],
    },
    "both-switch": {
        "label": "Both Switch",
        "tx_stages": [
            {"system": "You are a math problem solver. Solve step by step with full detailed reasoning."},
            {"system": "Continue with full detailed step-by-step reasoning. Show all work."},
            {"system": "Now compress your solution for a math expert recipient. Include only essential steps and the final answer. Use compact notation."},
        ],
        "rx_stages": [
            {"system": "You receive a compact mathematical message. Decompress it: interpret all notation and reconstruct the full reasoning."},
            {"system": "Based on your interpretation, verify the solution and state the final numeric answer. Output ONLY the number."},
        ],
    },
    "reverse": {
        "label": "Reverse",
        "tx_stages": [
            {"system": "Be concise. Outline the approach to solve this problem in minimal words."},
            {"system": "Now solve the problem in full detail. Show all steps and calculations."},
            {"system": "Present the complete detailed solution with the final answer clearly stated."},
        ],
        "rx_stages": [
            {"system": "Read the following solution freely and extract the final numeric answer. Output ONLY the number."},
            {"system": "Interpret this as a compact message. Extract the number. Output ONLY the number."},
        ],
    },
}

print(f"Defined {len(CONDITIONS)} conditions:")
for key, cond in CONDITIONS.items():
    print(f"  {cond['label']}: Tx {len(cond['tx_stages'])} stages, Rx {len(cond['rx_stages'])} stages")

In [ ]:
# ============================================================
# Trial Runner: Tx 3-stage chain -> transmit Stage 3 -> Rx
# ============================================================

def run_trial(problem, condition_key):
    """Run a single trial: Tx 3-stage CoT chain, then Rx extraction."""
    cond = CONDITIONS[condition_key]
    tx_stages = cond["tx_stages"]
    rx_stages = cond["rx_stages"]

    tx_tokens = {"prompt": 0, "completion": 0}
    rx_tokens = {"prompt": 0, "completion": 0}
    stage_outputs = []

    # Tx 3-stage chain
    prev_output = problem["text"]
    for i in range(3):
        user_msg = problem["text"] if i == 0 else f"Previous analysis:\n{prev_output}\n\nContinue."
        result = call_llm(tx_stages[i]["system"], user_msg)
        usage = result["usage"]
        tx_tokens["prompt"] += usage.get("prompt_tokens", 0)
        tx_tokens["completion"] += usage.get("completion_tokens", 0)
        stage_outputs.append(result["content"])
        prev_output = result["content"]

    # Transmitted message = Stage 3 output only
    transmitted = stage_outputs[2]
    msg_tokens = count_approx_tokens(transmitted)

    # Rx stages
    rx_prev = transmitted
    rx_output = ""
    for i in range(len(rx_stages)):
        user_msg = transmitted if i == 0 else f"Previous interpretation:\n{rx_prev}\n\nContinue."
        result = call_llm(rx_stages[i]["system"], user_msg)
        usage = result["usage"]
        rx_tokens["prompt"] += usage.get("prompt_tokens", 0)
        rx_tokens["completion"] += usage.get("completion_tokens", 0)
        rx_prev = result["content"]
        rx_output = result["content"]

    # Grade
    extracted = extract_number(rx_output)
    correct = False
    if problem.get("tolerance_fn") == "time":
        mins = parse_time_to_minutes(rx_output)
        correct = mins is not None and abs(mins - 300) < 15
    elif extracted is not None:
        tol = abs(problem["answer"]) * 0.10
        correct = abs(extracted - problem["answer"]) <= max(tol, 0.5)

    return {
        "problem_id": problem["id"],
        "condition": condition_key,
        "correct": correct,
        "expected": problem["answer"],
        "extracted": extracted,
        "rx_output": rx_output[:200],
        "msg_tokens": msg_tokens,
        "tx_tokens": tx_tokens["prompt"] + tx_tokens["completion"],
        "rx_tokens": rx_tokens["prompt"] + rx_tokens["completion"],
        "total_tokens": tx_tokens["prompt"] + tx_tokens["completion"] + rx_tokens["prompt"] + rx_tokens["completion"],
        "transmitted_msg": transmitted[:500],
    }

print("Trial runner loaded.")

## Condition 1: All General

Tx와 Rx 모두 일반적인 프롬프트를 사용합니다. 청중 인지 없음.
Tx는 전체 CoT에서 상세하게 풀고, Rx는 단순 추출만 수행합니다.

In [ ]:
# ============================================================
# Condition 1: All General
# ============================================================
print("Running Condition 1: All General...")
results_all_general = []
for p in PROBLEMS:
    print(f"  Problem {p['id']} [{p['category']}]")
    result = run_trial(p, "all-general")
    results_all_general.append(result)
    status = "OK" if result["correct"] else f"FAIL (got {result['extracted']})"
    print(f"    -> {status}, msg_tokens={result['msg_tokens']}")
    time.sleep(0.5)

acc = sum(1 for r in results_all_general if r["correct"]) / len(results_all_general) * 100
avg_msg = sum(r["msg_tokens"] for r in results_all_general) / len(results_all_general)
print(f"\nAll General: Accuracy={acc:.1f}%, Avg Msg Tokens={avg_msg:.0f}")

## Condition 2: All Audience-Aware

Tx의 모든 단계에서 수학 전문가를 대상으로 간결하게 작성합니다.
Rx도 전문가로서 압축된 메시지를 해석합니다.

In [ ]:
# ============================================================
# Condition 2: All Audience-Aware
# ============================================================
print("Running Condition 2: All Audience-Aware...")
results_all_audience = []
for p in PROBLEMS:
    print(f"  Problem {p['id']} [{p['category']}]")
    result = run_trial(p, "all-audience")
    results_all_audience.append(result)
    status = "OK" if result["correct"] else f"FAIL (got {result['extracted']})"
    print(f"    -> {status}, msg_tokens={result['msg_tokens']}")
    time.sleep(0.5)

acc = sum(1 for r in results_all_audience if r["correct"]) / len(results_all_audience) * 100
avg_msg = sum(r["msg_tokens"] for r in results_all_audience) / len(results_all_audience)
print(f"\nAll Audience-Aware: Accuracy={acc:.1f}%, Avg Msg Tokens={avg_msg:.0f}")

## Condition 3: Tx-Only Switch

Tx는 Stage 1-2에서 상세하게 풀고, Stage 3에서만 전문가 대상으로 압축합니다.
Rx는 일반 추출기입니다 (전문가 모드 없음).

In [ ]:
# ============================================================
# Condition 3: Tx-Only Switch
# ============================================================
print("Running Condition 3: Tx-Only Switch...")
results_tx_switch = []
for p in PROBLEMS:
    print(f"  Problem {p['id']} [{p['category']}]")
    result = run_trial(p, "tx-switch")
    results_tx_switch.append(result)
    status = "OK" if result["correct"] else f"FAIL (got {result['extracted']})"
    print(f"    -> {status}, msg_tokens={result['msg_tokens']}")
    time.sleep(0.5)

acc = sum(1 for r in results_tx_switch if r["correct"]) / len(results_tx_switch) * 100
avg_msg = sum(r["msg_tokens"] for r in results_tx_switch) / len(results_tx_switch)
print(f"\nTx-Only Switch: Accuracy={acc:.1f}%, Avg Msg Tokens={avg_msg:.0f}")

## Condition 4: Both Switch

Tx는 Stage 1-2에서 상세하게 풀고, Stage 3에서 압축합니다.
Rx는 2단계로 동작: 먼저 압축된 메시지를 복원한 후 검증하여 답을 추출합니다.

In [ ]:
# ============================================================
# Condition 4: Both Switch
# ============================================================
print("Running Condition 4: Both Switch...")
results_both_switch = []
for p in PROBLEMS:
    print(f"  Problem {p['id']} [{p['category']}]")
    result = run_trial(p, "both-switch")
    results_both_switch.append(result)
    status = "OK" if result["correct"] else f"FAIL (got {result['extracted']})"
    print(f"    -> {status}, msg_tokens={result['msg_tokens']}")
    time.sleep(0.5)

acc = sum(1 for r in results_both_switch if r["correct"]) / len(results_both_switch) * 100
avg_msg = sum(r["msg_tokens"] for r in results_both_switch) / len(results_both_switch)
print(f"\nBoth Switch: Accuracy={acc:.1f}%, Avg Msg Tokens={avg_msg:.0f}")

## Condition 5: Reverse

Tx는 Stage 1에서 간결한 개요를 작성하고, Stage 2-3에서 상세하게 확장합니다.
일반적 직관과 반대 순서로, 압축 효과가 떨어질 것으로 예상됩니다.

In [ ]:
# ============================================================
# Condition 5: Reverse
# ============================================================
print("Running Condition 5: Reverse...")
results_reverse = []
for p in PROBLEMS:
    print(f"  Problem {p['id']} [{p['category']}]")
    result = run_trial(p, "reverse")
    results_reverse.append(result)
    status = "OK" if result["correct"] else f"FAIL (got {result['extracted']})"
    print(f"    -> {status}, msg_tokens={result['msg_tokens']}")
    time.sleep(0.5)

acc = sum(1 for r in results_reverse if r["correct"]) / len(results_reverse) * 100
avg_msg = sum(r["msg_tokens"] for r in results_reverse) / len(results_reverse)
print(f"\nReverse: Accuracy={acc:.1f}%, Avg Msg Tokens={avg_msg:.0f}")

## 결과 요약

In [ ]:
# ============================================================
# Results Summary: Aggregation by condition + category
# ============================================================

all_results = (results_all_general + results_all_audience +
               results_tx_switch + results_both_switch + results_reverse)

# Build summary per condition
summary_rows = []
for cond_key, cond_info in CONDITIONS.items():
    cond_results = [r for r in all_results if r["condition"] == cond_key]
    correct = sum(1 for r in cond_results if r["correct"])
    total = len(cond_results)
    avg_msg = sum(r["msg_tokens"] for r in cond_results) / total
    avg_tx = sum(r["tx_tokens"] for r in cond_results) / total
    avg_rx = sum(r["rx_tokens"] for r in cond_results) / total
    avg_total = sum(r["total_tokens"] for r in cond_results) / total

    # By category
    by_cat = {}
    for cat in ["probability", "optimization", "multistep"]:
        cat_probs = [p["id"] for p in PROBLEMS if p["category"] == cat]
        cat_results = [r for r in cond_results if r["problem_id"] in cat_probs]
        cat_correct = sum(1 for r in cat_results if r["correct"])
        by_cat[cat] = f"{cat_correct}/{len(cat_results)}"

    summary_rows.append({
        "Condition": cond_info["label"],
        "Accuracy (%)": round(correct / total * 100, 1),
        "Correct": f"{correct}/{total}",
        "Avg Msg Tok": round(avg_msg),
        "Avg Tx Tok": round(avg_tx),
        "Avg Rx Tok": round(avg_rx),
        "Avg Total Tok": round(avg_total),
        "Probability": by_cat["probability"],
        "Optimization": by_cat["optimization"],
        "Multi-step": by_cat["multistep"],
    })

df_summary = pd.DataFrame(summary_rows)
print("=" * 100)
print("                       KI-3 RESULTS SUMMARY")
print("=" * 100)
display(df_summary)

# Per-problem detail
print("\n--- Per-Problem Detail ---")
per_problem = []
for p in PROBLEMS:
    row = {"ID": p["id"], "Category": p["category"], "Expected": p["answer"]}
    for cond_key, cond_info in CONDITIONS.items():
        r = [x for x in all_results if x["problem_id"] == p["id"] and x["condition"] == cond_key][0]
        row[cond_info["label"]] = "OK" if r["correct"] else f"FAIL({r['extracted']})"
    per_problem.append(row)

df_detail = pd.DataFrame(per_problem)
display(df_detail)

# Token usage
print(f"\nTotal API tokens: prompt={total_api_tokens['prompt']}, "
      f"completion={total_api_tokens['completion']}, "
      f"total={total_api_tokens['prompt'] + total_api_tokens['completion']}")

# Save results to JSON
output = {
    "experiment": "KI-3 Stage-Wise Switching",
    "model": MODEL,
    "temperature": TEMPERATURE,
    "timestamp": datetime.now().isoformat(),
    "total_api_tokens": total_api_tokens,
    "summary": summary_rows,
    "per_problem": per_problem,
    "results": [{k: v for k, v in r.items()} for r in all_results],
}

out_path = "ki3_final_results.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"\nResults saved to {out_path}")